# Fraunhofer's Discovery of Solar Absorption Lines (1814-1823)
# Fraunhofer의 태양 흡수선 발견 (1814-1823)

---

**Paper**: Joseph von Fraunhofer, *"Bestimmung des Brechungs- und Farbenzerstreuungs-Vermogens verschiedener Glasarten"* (1817)
and *"Neue Modifikation des Lichtes durch gegenseitige Einwirkung und Beugung der Strahlen"* (1823)

**핵심 기여 (Key Contributions)**:
- 태양 스펙트럼에서 574개 이상의 어두운 흡수선을 체계적으로 관측하고 기록 / Systematically observed and cataloged 574+ dark absorption lines in the solar spectrum
- 주요 선에 A-K 문자를 부여하여 표준 참조 체계 확립 / Assigned letters A-K to major lines, establishing a standard reference system
- 회절격자를 발명하여 파장의 절대 측정 가능하게 함 / Invented the diffraction grating, enabling absolute wavelength measurement
- 항성 스펙트럼도 관측하여 별마다 다른 흡수선 패턴 발견 / Observed stellar spectra, finding different line patterns for different stars

**이 노트북의 목표 (Notebook Goals)**:
1. Prism dispersion 시뮬레이션 (Snell's Law & Cauchy Equation)
2. Fraunhofer 흡수선이 포함된 태양 스펙트럼 시뮬레이션
3. Minimum deviation method 시연
4. 회절격자(diffraction grating) 시뮬레이션
5. Blackbody radiation과 흡수 물리학
6. 항성 스펙트럼 비교

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.patches as patches
from matplotlib.collections import LineCollection
from matplotlib.colors import LinearSegmentedColormap
from scipy.constants import h, c, k  # Planck, speed of light, Boltzmann

# Korean font setup for matplotlib.
plt.rcParams["font.family"] = "Apple SD Gothic Neo"
plt.rcParams["axes.unicode_minus"] = False

# High-quality figure defaults.
plt.rcParams["figure.dpi"] = 120
plt.rcParams["figure.figsize"] = (12, 6)
plt.rcParams["axes.grid"] = True
plt.rcParams["grid.alpha"] = 0.3


def wavelength_to_rgb(wavelength_nm: float, gamma: float = 0.8) -> tuple:
    """Convert a wavelength in nm to an RGB tuple.

    Args:
        wavelength_nm: Wavelength in nanometers (380-780 nm).
        gamma: Gamma correction factor.

    Returns:
        Tuple of (R, G, B) values in [0, 1].
    """
    wl = wavelength_nm
    if 380 <= wl < 440:
        r = -(wl - 440) / (440 - 380)
        g = 0.0
        b = 1.0
    elif 440 <= wl < 490:
        r = 0.0
        g = (wl - 440) / (490 - 440)
        b = 1.0
    elif 490 <= wl < 510:
        r = 0.0
        g = 1.0
        b = -(wl - 510) / (510 - 490)
    elif 510 <= wl < 580:
        r = (wl - 510) / (580 - 510)
        g = 1.0
        b = 0.0
    elif 580 <= wl < 645:
        r = 1.0
        g = -(wl - 645) / (645 - 580)
        b = 0.0
    elif 645 <= wl <= 780:
        r = 1.0
        g = 0.0
        b = 0.0
    else:
        r = g = b = 0.0

    # Intensity falloff at edges of visible spectrum.
    if 380 <= wl < 420:
        factor = 0.3 + 0.7 * (wl - 380) / (420 - 380)
    elif 420 <= wl <= 700:
        factor = 1.0
    elif 700 < wl <= 780:
        factor = 0.3 + 0.7 * (780 - wl) / (780 - 700)
    else:
        factor = 0.0

    r = (r * factor) ** gamma
    g = (g * factor) ** gamma
    b = (b * factor) ** gamma
    return (r, g, b)


# Fraunhofer line data: label, wavelength (nm), element, relative strength.
FRAUNHOFER_LINES = {
    "A":  {"wl": 759.37, "element": "$O_2$",  "strength": 0.6},
    "B":  {"wl": 686.72, "element": "$O_2$",  "strength": 0.7},
    "C":  {"wl": 656.28, "element": "H-alpha",    "strength": 0.8},
    "D1": {"wl": 589.59, "element": "Na",     "strength": 0.95},
    "D2": {"wl": 588.99, "element": "Na",     "strength": 0.95},
    "E":  {"wl": 527.04, "element": "Fe",     "strength": 0.6},
    "F":  {"wl": 486.13, "element": "H-beta",     "strength": 0.75},
    "G":  {"wl": 430.78, "element": "Fe/Ca",  "strength": 0.7},
    "H":  {"wl": 396.85, "element": "Ca II",  "strength": 0.9},
    "K":  {"wl": 393.37, "element": "Ca II",  "strength": 0.9},
}

print("Setup complete / 설정 완료")

---
## Part 1: Prism Dispersion -- Snell's Law & Cauchy Equation
## 파트 1: 프리즘 분산 -- 스넬의 법칙과 코시 방정식

Fraunhofer의 핵심 도구는 **유리 프리즘**이었습니다. 프리즘이 백색광을 스펙트럼으로 분리하는 원리를 이해하려면 두 가지 물리 법칙이 필요합니다:

**Snell's Law (스넬의 법칙)**:
$$n_1 \sin\theta_1 = n_2 \sin\theta_2$$

빛이 굴절률이 다른 매질의 경계면을 지날 때 방향이 바뀝니다.

**Cauchy's Equation (코시 방정식)**:
$$n(\lambda) = A + \frac{B}{\lambda^2} + \frac{C}{\lambda^4} + \cdots$$

굴절률은 파장에 따라 달라집니다. 짧은 파장(보라색)일수록 굴절률이 높아 더 많이 꺾입니다.
이것이 **분산(dispersion)**의 원리입니다.

Fraunhofer는 서로 다른 유리의 분산 특성을 정밀하게 측정하기 위해 흡수선을 기준점으로 사용했습니다. 이것이 그의 연구의 실용적 동기였습니다.

In [ ]:
# =============================================================================
# Part 1a: Cauchy equation -- refractive index vs wavelength
# =============================================================================

def cauchy_n(wavelength_nm: np.ndarray, A: float, B: float, C: float = 0.0) -> np.ndarray:
    """Calculate refractive index using Cauchy's equation.

    Args:
        wavelength_nm: Wavelength array in nanometers.
        A: Cauchy coefficient A (dimensionless).
        B: Cauchy coefficient B (nm^2).
        C: Cauchy coefficient C (nm^4).

    Returns:
        Refractive index array.
    """
    lam = wavelength_nm
    return A + B / lam**2 + C / lam**4


# Wavelength range: visible spectrum.
wavelengths = np.linspace(380, 780, 500)

# Cauchy coefficients for common optical glasses.
# Crown glass (BK7): lower dispersion.
n_crown = cauchy_n(wavelengths, A=1.5046, B=4200, C=0)

# Flint glass (SF11): higher dispersion.
n_flint = cauchy_n(wavelengths, A=1.7280, B=13400, C=0)

# --- Plot ---
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 5))

# Left: n(lambda) curves.
ax1.plot(wavelengths, n_crown, "b-", linewidth=2, label="Crown glass (BK7)")
ax1.plot(wavelengths, n_flint, "r-", linewidth=2, label="Flint glass (SF11)")

# Mark Fraunhofer D line as reference.
d_line = 589.3
ax1.axvline(d_line, color="orange", linestyle="--", alpha=0.7, label=f"D line ({d_line} nm)")

ax1.set_xlabel("Wavelength / 파장 (nm)", fontsize=12)
ax1.set_ylabel("Refractive Index / 굴절률 n", fontsize=12)
ax1.set_title("Cauchy Equation: n(lambda) for Optical Glasses\n코시 방정식: 광학 유리의 굴절률", fontsize=13)
ax1.legend(fontsize=10)
ax1.set_xlim(380, 780)

# Add color bar at bottom of left plot.
for wl_i in wavelengths:
    ax1.axvspan(wl_i - 0.5, wl_i + 0.5, ymin=0, ymax=0.03,
                color=wavelength_to_rgb(wl_i), alpha=0.8)

# Right: Dispersion dn/dlambda.
dn_crown = np.gradient(n_crown, wavelengths)
dn_flint = np.gradient(n_flint, wavelengths)

ax2.plot(wavelengths, -dn_crown * 1e3, "b-", linewidth=2, label="Crown glass (BK7)")
ax2.plot(wavelengths, -dn_flint * 1e3, "r-", linewidth=2, label="Flint glass (SF11)")
ax2.set_xlabel("Wavelength / 파장 (nm)", fontsize=12)
ax2.set_ylabel(r"$-dn/d\lambda$ ($\times 10^{-3}$ nm$^{-1}$)", fontsize=12)
ax2.set_title("Dispersion: Stronger at Shorter Wavelengths\n분산: 짧은 파장에서 더 강함", fontsize=13)
ax2.legend(fontsize=10)
ax2.set_xlim(380, 780)

plt.tight_layout()
plt.show()

print(f"\nCrown glass: n(400nm) = {cauchy_n(400, 1.5046, 4200):.4f}, "
      f"n(700nm) = {cauchy_n(700, 1.5046, 4200):.4f}")
print(f"Flint glass: n(400nm) = {cauchy_n(400, 1.7280, 13400):.4f}, "
      f"n(700nm) = {cauchy_n(700, 1.7280, 13400):.4f}")
print("=> Flint glass가 더 큰 분산(dispersion)을 보임")

In [ ]:
# =============================================================================
# Part 1b: Prism dispersion visualization
# =============================================================================

def prism_deviation(n: float, apex_angle_deg: float = 60.0) -> float:
    """Calculate minimum deviation angle for a prism.

    Args:
        n: Refractive index.
        apex_angle_deg: Prism apex angle in degrees.

    Returns:
        Minimum deviation angle in degrees.
    """
    alpha = np.radians(apex_angle_deg)
    delta_min = 2 * np.arcsin(n * np.sin(alpha / 2)) - alpha
    return np.degrees(delta_min)


fig, ax = plt.subplots(figsize=(12, 7))

# Draw prism (triangle).
prism_x = [3, 5, 4, 3]
prism_y = [1, 1, 3.5, 1]
prism = plt.Polygon(list(zip(prism_x, prism_y)), fill=True,
                     facecolor="lightblue", edgecolor="black", linewidth=2, alpha=0.4)
ax.add_patch(prism)
ax.text(4, 1.8, "Prism\n(60 deg)", ha="center", fontsize=11, fontweight="bold")

# Incident white light ray.
ax.annotate("", xy=(3.35, 2.2), xytext=(1, 2.2),
            arrowprops=dict(arrowstyle="->, head_width=0.15", color="black", lw=2))
ax.text(1.5, 2.45, "White light\n(백색광)", fontsize=10, ha="center")

# Dispersed rays exiting the prism.
n_rays = 7
colors_wl = np.linspace(380, 700, n_rays)
base_angle = -15  # degrees
spread = 25  # total angular spread

for i, wl in enumerate(colors_wl):
    angle = np.radians(base_angle + spread * i / (n_rays - 1))
    x_start = 4.75
    y_start = 1.45 + 0.15 * i / (n_rays - 1)
    length = 2.5
    x_end = x_start + length * np.cos(angle)
    y_end = y_start + length * np.sin(angle)
    color = wavelength_to_rgb(wl)
    ax.annotate("", xy=(x_end, y_end), xytext=(x_start, y_start),
                arrowprops=dict(arrowstyle="->, head_width=0.08", color=color, lw=2.5))

# Labels for red and violet ends.
ax.text(7.5, 1.05, "Red (700 nm)\n(빨간색, 적은 굴절)", fontsize=9, color="red",
        ha="center")
ax.text(7.0, 2.45, "Violet (380 nm)\n(보라색, 큰 굴절)", fontsize=9, color="purple",
        ha="center")

# Annotations.
ax.text(4, 4.0,
        "Shorter wavelength -> higher n -> more refraction\n"
        "짧은 파장 -> 높은 굴절률 -> 더 큰 굴절",
        fontsize=11, ha="center", style="italic",
        bbox=dict(boxstyle="round,pad=0.3", facecolor="lightyellow", alpha=0.8))

ax.set_xlim(0.5, 8.5)
ax.set_ylim(0.3, 4.5)
ax.set_aspect("equal")
ax.axis("off")
ax.set_title("Prism Dispersion of White Light\n프리즘에 의한 백색광 분산", fontsize=14)
plt.tight_layout()
plt.show()

---
## Part 2: Simulating the Solar Spectrum with Absorption Lines
## 파트 2: 흡수선을 포함한 태양 스펙트럼 시뮬레이션

Fraunhofer가 발견한 것은 **연속 스펙트럼 위에 놓인 어두운 선(dark lines)**이었습니다.

**물리적 원리**:
1. 태양의 광구(photosphere)는 뜨거운 기체로, 연속적인 **blackbody spectrum**을 방출합니다
2. 이 빛이 태양 대기(chromosphere)를 통과하면서, 대기의 원소들이 **특정 파장의 빛을 흡수**합니다
3. 각 원소는 고유한 에너지 준위 구조를 가지므로, 고유한 파장에서 흡수가 일어납니다
4. 결과: 연속 스펙트럼 위에 어두운 **흡수선(absorption lines)**이 나타남

Fraunhofer는 가장 뚜렷한 선들에 알파벳 문자 A부터 K까지 붙였습니다.
이 명명법은 오늘날까지도 사용됩니다 (예: Na D 선, Ca H & K 선).

In [ ]:
# =============================================================================
# Part 2: Solar spectrum with Fraunhofer absorption lines
# =============================================================================

def planck_wavelength(wavelength_m: np.ndarray, T: float) -> np.ndarray:
    """Calculate Planck spectral radiance B(lambda, T).

    Args:
        wavelength_m: Wavelength array in meters.
        T: Temperature in Kelvin.

    Returns:
        Spectral radiance in W / (m^2 sr m).
    """
    return (2 * h * c**2 / wavelength_m**5) / (np.exp(h * c / (wavelength_m * k * T)) - 1)


def add_absorption_lines(wavelengths_nm: np.ndarray,
                         spectrum: np.ndarray,
                         lines: dict,
                         width_nm: float = 1.0) -> np.ndarray:
    """Add Gaussian absorption lines to a spectrum.

    Args:
        wavelengths_nm: Wavelength array in nm.
        spectrum: Continuum spectrum array.
        lines: Dictionary of Fraunhofer lines with 'wl' and 'strength'.
        width_nm: Gaussian width (sigma) of each line in nm.

    Returns:
        Spectrum with absorption lines.
    """
    absorbed = spectrum.copy()
    for label, info in lines.items():
        wl0 = info["wl"]
        strength = info["strength"]
        # Gaussian absorption profile.
        profile = strength * np.exp(-0.5 * ((wavelengths_nm - wl0) / width_nm)**2)
        absorbed *= (1 - profile)
    return absorbed


# Generate solar spectrum.
wl_nm = np.linspace(350, 800, 3000)
wl_m = wl_nm * 1e-9

# Blackbody continuum at T_sun.
T_sun = 5778  # K
bb_spectrum = planck_wavelength(wl_m, T_sun)
bb_normalized = bb_spectrum / bb_spectrum.max()

# Add Fraunhofer absorption lines.
solar_spectrum = add_absorption_lines(wl_nm, bb_normalized, FRAUNHOFER_LINES, width_nm=0.8)

# --- Plot ---
fig, axes = plt.subplots(3, 1, figsize=(14, 12), gridspec_kw={"height_ratios": [1, 3, 1]})

# Top: color bar showing continuous spectrum (no lines).
ax_top = axes[0]
for i in range(len(wl_nm) - 1):
    ax_top.axvspan(wl_nm[i], wl_nm[i+1], color=wavelength_to_rgb(wl_nm[i]),
                   alpha=bb_normalized[i])
ax_top.set_xlim(350, 800)
ax_top.set_yticks([])
ax_top.set_title("Continuous Blackbody Spectrum (No Absorption)\n"
                  "연속 흑체 스펙트럼 (흡수 없음)", fontsize=13)

# Middle: spectrum plot with absorption lines.
ax_mid = axes[1]
ax_mid.fill_between(wl_nm, solar_spectrum, alpha=0.3, color="gray")
ax_mid.plot(wl_nm, bb_normalized, "k--", alpha=0.4, linewidth=1, label="Blackbody continuum")
ax_mid.plot(wl_nm, solar_spectrum, "k-", linewidth=0.5, label="Solar spectrum with absorption")

# Label each Fraunhofer line.
for label, info in FRAUNHOFER_LINES.items():
    wl0 = info["wl"]
    elem = info["element"]
    if 350 <= wl0 <= 800:
        ax_mid.axvline(wl0, color="red", alpha=0.5, linewidth=0.8, linestyle="--")
        # Stagger label heights to avoid overlap.
        y_pos = 1.05 if label in ["A", "C", "E", "G", "K"] else 1.12
        ax_mid.annotate(f"{label}\n{wl0:.1f} nm\n{elem}",
                        xy=(wl0, 0), xytext=(wl0, y_pos),
                        fontsize=8, ha="center", va="bottom",
                        color="red",
                        arrowprops=dict(arrowstyle="-", color="red", alpha=0.3))

ax_mid.set_xlim(350, 800)
ax_mid.set_ylim(0, 1.25)
ax_mid.set_xlabel("Wavelength / 파장 (nm)", fontsize=12)
ax_mid.set_ylabel("Normalized Intensity\n정규화 강도", fontsize=12)
ax_mid.set_title("Solar Spectrum with Fraunhofer Absorption Lines\n"
                  "Fraunhofer 흡수선이 포함된 태양 스펙트럼", fontsize=14)
ax_mid.legend(fontsize=10, loc="upper right")

# Bottom: color bar showing spectrum WITH absorption lines (dark lines visible).
ax_bot = axes[2]
for i in range(len(wl_nm) - 1):
    rgb = wavelength_to_rgb(wl_nm[i])
    intensity = solar_spectrum[i]
    darkened = tuple(ch * intensity for ch in rgb)
    ax_bot.axvspan(wl_nm[i], wl_nm[i+1], color=darkened)
ax_bot.set_xlim(350, 800)
ax_bot.set_yticks([])
ax_bot.set_title("Fraunhofer's Solar Spectrum (Dark Lines Visible)\n"
                  "Fraunhofer의 태양 스펙트럼 (어두운 선이 보임)", fontsize=13)
ax_bot.set_xlabel("Wavelength / 파장 (nm)", fontsize=12)

plt.tight_layout()
plt.show()

print("\n=== Fraunhofer Lines Table / Fraunhofer 선 목록 ===")
print(f"{'Label':<6} {'Wavelength (nm)':<18} {'Element':<12} {'Strength':<10}")
print("-" * 50)
for label, info in FRAUNHOFER_LINES.items():
    print(f"{label:<6} {info['wl']:<18.2f} {info['element']:<12} {info['strength']:<10.2f}")

---
## Part 3: Fraunhofer's Measurement -- Minimum Deviation Method
## 파트 3: Fraunhofer의 측정 -- 최소 편향각 방법

Fraunhofer는 각 흡수선의 파장을 정밀하게 측정하기 위해 **최소 편향각(minimum deviation)** 방법을 사용했습니다.

**원리**: 프리즘을 통과하는 빛의 편향각(deviation angle) $\delta$는 입사각에 따라 변하지만,
특정 조건에서 **최솟값** $\delta_{\min}$을 가집니다. 이때 빛은 프리즘 내부에서 밑면에 평행하게 진행합니다.

**핵심 공식**:
$$n = \frac{\sin\left(\frac{\alpha + \delta_{\min}}{2}\right)}{\sin\left(\frac{\alpha}{2}\right)}$$

여기서:
- $n$ = 굴절률 (refractive index)
- $\alpha$ = 프리즘의 꼭지각 (apex angle)
- $\delta_{\min}$ = 최소 편향각 (minimum deviation angle)

이 방법의 장점:
1. $\delta_{\min}$에서는 미세한 각도 변화에 둔감 (안정적 측정)
2. 하나의 측정으로 굴절률을 정밀하게 결정 가능
3. 각 흡수선에서 $\delta_{\min}$을 측정하면 해당 파장의 $n$을 알 수 있음

In [ ]:
# =============================================================================
# Part 3a: Deviation angle vs incident angle (showing minimum)
# =============================================================================

def deviation_angle(theta_i_deg: float, n: float, alpha_deg: float = 60.0) -> float:
    """Calculate deviation angle for a prism at given incidence angle.

    Args:
        theta_i_deg: Incident angle in degrees.
        n: Refractive index of prism.
        alpha_deg: Apex angle of prism in degrees.

    Returns:
        Deviation angle in degrees, or NaN if total internal reflection.
    """
    alpha = np.radians(alpha_deg)
    theta_i = np.radians(theta_i_deg)

    # Refraction at first surface.
    sin_r1 = np.sin(theta_i) / n
    if np.any(np.abs(sin_r1) > 1):
        return np.nan
    r1 = np.arcsin(sin_r1)

    # Angle at second surface.
    r2 = alpha - r1

    # Refraction at second surface.
    sin_theta_t = n * np.sin(r2)
    if np.any(np.abs(sin_theta_t) > 1):
        return np.nan
    theta_t = np.arcsin(sin_theta_t)

    # Total deviation.
    delta = theta_i + theta_t - alpha
    return np.degrees(delta)


# Parameters.
alpha_deg = 60.0  # Prism apex angle.
n_d = 1.515       # Crown glass at D line.

# Incident angle range.
theta_range = np.linspace(30, 80, 200)
delta_values = []
for th in theta_range:
    d = deviation_angle(th, n_d, alpha_deg)
    delta_values.append(d)
delta_values = np.array(delta_values)

# Find minimum deviation.
valid = ~np.isnan(delta_values)
delta_min_idx = np.nanargmin(delta_values)
theta_min = theta_range[delta_min_idx]
delta_min = delta_values[delta_min_idx]

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 5))

# Left: deviation vs incident angle.
ax1.plot(theta_range[valid], delta_values[valid], "b-", linewidth=2)
ax1.axhline(delta_min, color="r", linestyle="--", alpha=0.7,
            label=f"$\\delta_{{\\min}}$ = {delta_min:.2f} deg")
ax1.axvline(theta_min, color="g", linestyle="--", alpha=0.7,
            label=f"$\\theta_i$ at min = {theta_min:.1f} deg")
ax1.plot(theta_min, delta_min, "ro", markersize=10, zorder=5)

ax1.set_xlabel("Incident Angle / 입사각 (deg)", fontsize=12)
ax1.set_ylabel("Deviation Angle / 편향각 $\\delta$ (deg)", fontsize=12)
ax1.set_title("Deviation vs Incident Angle (60 deg prism, n=1.515)\n"
              "편향각 vs 입사각", fontsize=13)
ax1.legend(fontsize=10)

# Verify formula: n = sin((alpha + delta_min)/2) / sin(alpha/2).
n_measured = np.sin(np.radians((alpha_deg + delta_min) / 2)) / np.sin(np.radians(alpha_deg / 2))
ax1.text(55, delta_min + 5,
         f"Verification:\n"
         f"n = sin(({alpha_deg:.0f} + {delta_min:.2f})/2) / sin({alpha_deg:.0f}/2)\n"
         f"n = {n_measured:.4f}  (input: {n_d})",
         fontsize=9, bbox=dict(boxstyle="round", facecolor="lightyellow", alpha=0.8))

# Right: delta_min vs wavelength for crown glass.
wl_plot = np.linspace(380, 780, 200)
n_plot = cauchy_n(wl_plot, A=1.5046, B=4200)
delta_min_vs_wl = np.array([prism_deviation(ni, alpha_deg) for ni in n_plot])

ax2.plot(wl_plot, delta_min_vs_wl, "b-", linewidth=2)

# Mark Fraunhofer lines on the curve.
for label, info in FRAUNHOFER_LINES.items():
    wl0 = info["wl"]
    if 380 <= wl0 <= 780:
        n_line = cauchy_n(wl0, A=1.5046, B=4200)
        d_line_val = prism_deviation(n_line, alpha_deg)
        ax2.plot(wl0, d_line_val, "ro", markersize=6)
        ax2.annotate(label, (wl0, d_line_val), textcoords="offset points",
                     xytext=(5, 5), fontsize=9, color="red")

# Add color strip at bottom.
for wl_i in wl_plot:
    ax2.axvspan(wl_i - 1, wl_i + 1, ymin=0, ymax=0.03,
                color=wavelength_to_rgb(wl_i), alpha=0.8)

ax2.set_xlabel("Wavelength / 파장 (nm)", fontsize=12)
ax2.set_ylabel("$\\delta_{\\min}$ (deg)", fontsize=12)
ax2.set_title("Minimum Deviation vs Wavelength (Crown Glass)\n"
              "최소 편향각 vs 파장", fontsize=13)

plt.tight_layout()
plt.show()

print(f"\nMinimum deviation angle (D line): {delta_min:.2f} deg")
print(f"Recovered n from delta_min: {n_measured:.4f} (true: {n_d})")
print("=> Fraunhofer는 이 방법으로 각 흡수선의 정확한 굴절률을 측정함")

In [ ]:
# =============================================================================
# Part 3b: Geometry of minimum deviation through a prism
# =============================================================================

fig, ax = plt.subplots(figsize=(12, 7))

# Draw prism.
apex = np.array([5, 6])
left = np.array([2, 1])
right = np.array([8, 1])
prism_tri = plt.Polygon([apex, left, right], fill=True,
                         facecolor="lightcyan", edgecolor="black", linewidth=2, alpha=0.5)
ax.add_patch(prism_tri)

# Label apex angle.
ax.annotate(r"$\alpha$ = 60 deg", xy=apex, xytext=(5, 6.5),
            fontsize=12, ha="center", fontweight="bold")

# Minimum deviation ray path: symmetric through prism.
# Entry point on left face, exit on right face.
entry = np.array([3.2, 3.0])
midpoint = np.array([5.0, 2.5])
exit_pt = np.array([6.8, 3.0])

# Incoming ray.
incoming_start = np.array([0.5, 4.5])
ax.annotate("", xy=entry, xytext=incoming_start,
            arrowprops=dict(arrowstyle="->, head_width=0.15", color="orange", lw=2.5))
ax.text(1.2, 4.2, "Incident ray\n입사광", fontsize=10, color="darkorange")

# Ray inside prism (parallel to base at min deviation).
ax.annotate("", xy=exit_pt, xytext=entry,
            arrowprops=dict(arrowstyle="->, head_width=0.1", color="orange", lw=2,
                          linestyle="dashed"))
ax.text(5.0, 2.0, "Parallel to base\n밑변에 평행", fontsize=9, ha="center",
        color="darkorange", style="italic")

# Outgoing ray.
outgoing_end = np.array([9.5, 4.5])
ax.annotate("", xy=outgoing_end, xytext=exit_pt,
            arrowprops=dict(arrowstyle="->, head_width=0.15", color="red", lw=2.5))
ax.text(9.0, 4.2, "Deviated ray\n편향된 빛", fontsize=10, color="red")

# Draw normals at entry and exit.
# Left face normal (perpendicular to left face).
left_face_dir = apex - left
left_normal = np.array([left_face_dir[1], -left_face_dir[0]])
left_normal = left_normal / np.linalg.norm(left_normal)
norm_len = 1.5
ax.plot([entry[0] - left_normal[0]*norm_len, entry[0] + left_normal[0]*norm_len],
        [entry[1] - left_normal[1]*norm_len, entry[1] + left_normal[1]*norm_len],
        "k--", alpha=0.4, linewidth=1)
ax.text(entry[0] - left_normal[0]*1.7, entry[1] - left_normal[1]*1.7 + 0.2,
        "Normal", fontsize=8, color="gray")

# Undeviated ray (extension of incoming).
direction = entry - incoming_start
direction = direction / np.linalg.norm(direction)
undev_end = entry + direction * 6
ax.plot([entry[0], undev_end[0]], [entry[1], undev_end[1]],
        "k:", alpha=0.3, linewidth=1)

# Deviation angle arc.
ax.annotate("", xy=(8.5, 3.5), xytext=(8.5, 4.2),
            arrowprops=dict(arrowstyle="<->", color="blue", lw=1.5))
ax.text(9.0, 3.8, r"$\delta_{\min}$", fontsize=14, color="blue", fontweight="bold")

# Formula box.
ax.text(5, 7.5,
        r"$n = \frac{\sin\left(\frac{\alpha + \delta_{\min}}{2}\right)}"
        r"{\sin\left(\frac{\alpha}{2}\right)}$",
        fontsize=16, ha="center",
        bbox=dict(boxstyle="round,pad=0.5", facecolor="lightyellow", edgecolor="black", alpha=0.9))

# Angle labels at entry.
from matplotlib.patches import Arc
arc1 = Arc(entry, 1.5, 1.5, angle=0, theta1=60, theta2=115, color="green", lw=1.5)
ax.add_patch(arc1)
ax.text(entry[0] - 0.5, entry[1] + 0.9, r"$\theta_1$", fontsize=11, color="green")

arc2 = Arc(entry, 1.0, 1.0, angle=0, theta1=115, theta2=155, color="purple", lw=1.5)
ax.add_patch(arc2)
ax.text(entry[0] - 0.9, entry[1] + 0.5, r"$r_1$", fontsize=11, color="purple")

ax.set_xlim(-0.5, 10.5)
ax.set_ylim(0, 8.5)
ax.set_aspect("equal")
ax.axis("off")
ax.set_title("Minimum Deviation Geometry / 최소 편향각 기하학\n"
             "At minimum deviation: $\\theta_1 = \\theta_2$ and $r_1 = r_2$ (symmetric path)",
             fontsize=13)
plt.tight_layout()
plt.show()

---
## Part 4: Diffraction Grating -- Fraunhofer's Innovation
## 파트 4: 회절격자 -- Fraunhofer의 혁신

Fraunhofer의 가장 중요한 기술적 혁신은 **회절격자(diffraction grating)**의 발명입니다 (1821).

**원리**: 평행한 슬릿 배열에 빛이 통과하면, 각 슬릿에서 나온 파동이 간섭합니다.
보강 간섭(constructive interference) 조건:

$$d \sin\theta = m\lambda$$

여기서:
- $d$ = 격자 간격 (grating spacing)
- $\theta$ = 회절각 (diffraction angle)
- $m$ = 차수 (order: 0, 1, 2, ...)
- $\lambda$ = 파장

**프리즘 대비 회절격자의 장점**:
1. **균일한 분산 (linear dispersion)**: 프리즘은 보라색 끝에서 분산이 크고 빨간색 끝에서 작지만,
   격자는 모든 파장에서 거의 균일한 각도 분산을 제공
2. **절대 파장 측정**: $d$와 $\theta$를 알면 $\lambda$를 직접 계산 가능
3. **분해능 (resolving power)**: $R = mN$ (차수 $m$ x 슬릿 수 $N$)

Fraunhofer는 격자를 사용해 흡수선의 절대 파장을 최초로 측정했습니다.
그의 측정값은 현대 값과 놀랍도록 일치합니다.

In [ ]:
# =============================================================================
# Part 4a: Diffraction grating simulation
# =============================================================================

def grating_intensity(theta_deg: np.ndarray, wavelength_nm: float,
                      d_nm: float, N: int) -> np.ndarray:
    """Calculate diffraction grating intensity pattern.

    Uses the multi-slit interference formula:
    I = (sin(N * beta) / sin(beta))^2
    where beta = pi * d * sin(theta) / lambda.

    Args:
        theta_deg: Array of angles in degrees.
        wavelength_nm: Wavelength in nm.
        d_nm: Grating spacing in nm.
        N: Number of slits.

    Returns:
        Normalized intensity array.
    """
    theta = np.radians(theta_deg)
    beta = np.pi * d_nm * np.sin(theta) / wavelength_nm

    # Avoid division by zero.
    with np.errstate(divide="ignore", invalid="ignore"):
        intensity = (np.sin(N * beta) / np.sin(beta))**2
        # At beta = m*pi, the value is N^2.
        intensity = np.where(np.abs(np.sin(beta)) < 1e-10, N**2, intensity)

    return intensity / N**2  # Normalize to max=1.


# Grating parameters.
d_um = 2.0              # Grating spacing: 2 micrometers (500 lines/mm).
d_nm = d_um * 1000      # Convert to nm.
N_slits = 100           # Number of slits.

# Angle range.
theta = np.linspace(-30, 30, 10000)

fig, (ax1, ax2) = plt.subplots(2, 1, figsize=(14, 10))

# Top: Intensity pattern for a single wavelength (Na D line).
wl_D = 589.3
I_D = grating_intensity(theta, wl_D, d_nm, N_slits)

ax1.plot(theta, I_D, "darkorange", linewidth=1)
ax1.fill_between(theta, I_D, alpha=0.3, color="orange")

# Mark diffraction orders.
for m in range(-2, 3):
    if m == 0:
        continue
    theta_m = np.degrees(np.arcsin(m * wl_D / d_nm))
    ax1.axvline(theta_m, color="red", linestyle="--", alpha=0.5)
    ax1.text(theta_m, 1.05, f"m={m}\n$\\theta$={theta_m:.1f} deg",
             ha="center", fontsize=9, color="red")

ax1.axvline(0, color="red", linestyle="--", alpha=0.5)
ax1.text(0, 1.05, "m=0", ha="center", fontsize=9, color="red")

ax1.set_xlabel("Angle / 각도 (deg)", fontsize=12)
ax1.set_ylabel("Normalized Intensity\n정규화 강도", fontsize=12)
ax1.set_title(f"Diffraction Grating Pattern (N={N_slits} slits, d={d_um} um, "
              f"lambda={wl_D} nm)\n"
              f"회절격자 패턴 ($d \\sin\\theta = m\\lambda$)", fontsize=13)
ax1.set_xlim(-30, 30)
ax1.set_ylim(0, 1.15)

# Bottom: Multiple wavelengths showing spectral separation in 1st order.
wavelengths_show = [400, 450, 500, 550, 589.3, 650, 700]
ax2.set_xlim(5, 25)

for wl in wavelengths_show:
    I_wl = grating_intensity(theta, wl, d_nm, N_slits)
    color = wavelength_to_rgb(wl)
    ax2.plot(theta, I_wl, color=color, linewidth=1.5, alpha=0.8)
    ax2.fill_between(theta, I_wl, alpha=0.2, color=color)

    # Label the 1st order peak.
    theta_1 = np.degrees(np.arcsin(wl / d_nm))
    ax2.text(theta_1, 1.03, f"{wl:.0f}", ha="center", fontsize=8, color=color)

ax2.set_xlabel("Angle / 각도 (deg)", fontsize=12)
ax2.set_ylabel("Normalized Intensity\n정규화 강도", fontsize=12)
ax2.set_title("1st Order Spectrum: Different Wavelengths Separated by Angle\n"
              "1차 스펙트럼: 파장별 각도 분리 (균일한 분산)", fontsize=13)
ax2.set_ylim(0, 1.15)

plt.tight_layout()
plt.show()

In [ ]:
# =============================================================================
# Part 4b: Prism vs Grating dispersion comparison + Resolving power
# =============================================================================

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 5))

# --- Left: Prism vs Grating angular dispersion ---
wl_range = np.linspace(380, 780, 200)

# Prism: angular position proportional to delta_min(lambda).
n_prism = cauchy_n(wl_range, A=1.5046, B=4200)
delta_prism = np.array([prism_deviation(ni, 60.0) for ni in n_prism])
# Normalize to same range for comparison.
delta_prism_norm = (delta_prism - delta_prism.min()) / (delta_prism.max() - delta_prism.min())

# Grating: theta = arcsin(m * lambda / d), 1st order.
d_nm_comp = 2000  # 2 um spacing.
theta_grating = np.degrees(np.arcsin(wl_range / d_nm_comp))
theta_grating_norm = (theta_grating - theta_grating.min()) / (theta_grating.max() - theta_grating.min())

ax1.plot(wl_range, delta_prism_norm, "b-", linewidth=2, label="Prism (nonlinear)")
ax1.plot(wl_range, theta_grating_norm, "r-", linewidth=2, label="Grating (nearly linear)")

ax1.set_xlabel("Wavelength / 파장 (nm)", fontsize=12)
ax1.set_ylabel("Normalized Angular Position\n정규화 각도 위치", fontsize=12)
ax1.set_title("Prism vs Grating: Angular Dispersion\n프리즘 vs 격자: 각도 분산 비교", fontsize=13)
ax1.legend(fontsize=10)

# Add color bar.
for wl_i in wl_range:
    ax1.axvspan(wl_i - 1, wl_i + 1, ymin=0, ymax=0.03,
                color=wavelength_to_rgb(wl_i), alpha=0.8)

# Annotate key difference.
ax1.annotate("Prism: compressed red,\nstretched violet\n프리즘: 빨간색 압축, 보라색 확장",
             xy=(450, 0.85), fontsize=9,
             bbox=dict(boxstyle="round", facecolor="lightblue", alpha=0.7))
ax1.annotate("Grating: uniform spacing\n격자: 균일한 간격",
             xy=(550, 0.35), fontsize=9,
             bbox=dict(boxstyle="round", facecolor="lightyellow", alpha=0.7))

# --- Right: Resolving power ---
N_values = [100, 500, 1000, 5000, 10000]
orders = [1, 2, 3]

ax2.set_xlabel("Number of Slits N / 슬릿 수", fontsize=12)
ax2.set_ylabel("Resolving Power R = mN\n분해능", fontsize=12)
ax2.set_title("Grating Resolving Power\n격자 분해능 R = mN", fontsize=13)

for m in orders:
    R = [m * N for N in N_values]
    ax2.plot(N_values, R, "o-", linewidth=2, markersize=6, label=f"m = {m}")

# Mark the resolving power needed to separate Na D1 and D2.
# Na D: 589.59 and 588.99 nm => Delta_lambda = 0.6 nm
# R_needed = lambda / Delta_lambda = 589.3 / 0.6 ~ 982.
R_na = 589.3 / 0.6
ax2.axhline(R_na, color="gray", linestyle="--", alpha=0.7)
ax2.text(500, R_na + 200,
         f"R ~ {R_na:.0f} needed to resolve Na D1/D2\n"
         f"Na D1/D2 분리에 필요한 분해능",
         fontsize=9, color="gray")

ax2.set_xscale("log")
ax2.set_yscale("log")
ax2.legend(fontsize=10)

plt.tight_layout()
plt.show()

# Summary comparison table.
print("\n=== Prism vs Grating Comparison / 프리즘 vs 격자 비교 ===")
print(f"{'Property':<30} {'Prism':<25} {'Grating':<25}")
print("-" * 80)
print(f"{'Dispersion type':<30} {'Nonlinear':<25} {'Nearly linear':<25}")
print(f"{'Wavelength measurement':<30} {'Indirect (via n)':<25} {'Direct (d*sin(theta))':<25}")
print(f"{'Resolving power':<30} {'Limited by glass':<25} {'R = mN (scalable)':<25}")
print(f"{'Efficiency':<30} {'High (one beam)':<25} {'Split into orders':<25}")
print(f"{'Fraunhofer used':<30} {'Yes (early work)':<25} {'Yes (1821 onward)':<25}")

---
## Part 5: Blackbody Radiation & Absorption Physics
## 파트 5: 흑체 복사와 흡수 물리학

Fraunhofer의 시대에는 흡수선의 물리적 원인이 알려지지 않았습니다.
Kirchhoff & Bunsen (1859-1860)이 원소의 방출/흡수 스펙트럼의 관계를 밝혀낸 것은 약 40년 후입니다.

**현대적 이해**:

1. **Planck 함수 (흑체 복사)**:
$$B(\lambda, T) = \frac{2hc^2}{\lambda^5} \cdot \frac{1}{e^{hc/\lambda k_B T} - 1}$$

태양 광구(~5778 K)는 근사적으로 흑체 스펙트럼을 방출합니다.

2. **Beer-Lambert Law (흡수 법칙)**:
$$I(\lambda) = I_0(\lambda) \cdot e^{-\tau(\lambda)}$$

빛이 흡수 매질을 통과할 때 강도가 지수적으로 감소합니다.

3. **Optical depth (광학 깊이)** $\tau(\lambda)$:
- 특정 원소의 전이(transition) 파장에서 $\tau$가 급격히 증가
- 이것이 스펙트럼에 어두운 선으로 나타남
- 각 원소는 고유한 에너지 준위 구조 -> 고유한 $\tau(\lambda)$ 패턴

In [ ]:
# =============================================================================
# Part 5a: Planck function for the Sun
# =============================================================================

wl_um = np.linspace(0.1, 3.0, 1000)  # Wavelength in micrometers.
wl_m_planck = wl_um * 1e-6

# Solar temperature.
T_sun = 5778  # K

# Planck function.
B_sun = planck_wavelength(wl_m_planck, T_sun)

# Wien's displacement law: lambda_max * T = 2.898e-3 m K.
lambda_max = 2.898e-3 / T_sun * 1e6  # in micrometers

fig, ax = plt.subplots(figsize=(12, 6))

ax.plot(wl_um, B_sun / 1e13, "k-", linewidth=2, label=f"T = {T_sun} K (Sun)")

# Also show blackbodies at other temperatures for context.
for T, color, name in [(3500, "red", "M star (~3500 K)"),
                        (7500, "blue", "A star (~7500 K)"),
                        (10000, "purple", "B star (~10000 K)")]:
    B_T = planck_wavelength(wl_m_planck, T)
    ax.plot(wl_um, B_T / 1e13, color=color, linewidth=1.5, linestyle="--",
            alpha=0.7, label=name)

ax.axvline(lambda_max, color="orange", linestyle=":", alpha=0.7)
ax.text(lambda_max + 0.05, B_sun.max() / 1e13 * 0.95,
        f"Wien peak\n$\\lambda_{{max}}$ = {lambda_max:.3f} um\n({lambda_max*1000:.0f} nm)",
        fontsize=10, color="orange")

# Shade visible region.
ax.axvspan(0.38, 0.78, alpha=0.1, color="yellow", label="Visible / 가시광선")

ax.set_xlabel("Wavelength / 파장 ($\\mu$m)", fontsize=12)
ax.set_ylabel("$B(\\lambda, T)$ ($\\times 10^{13}$ W m$^{-2}$ sr$^{-1}$ m$^{-1}$)", fontsize=12)
ax.set_title("Planck Function: Blackbody Spectra at Different Temperatures\n"
             "Planck 함수: 다른 온도에서의 흑체 스펙트럼", fontsize=14)
ax.legend(fontsize=10)
ax.set_xlim(0.1, 3.0)
ax.set_ylim(0, None)

plt.tight_layout()
plt.show()

print(f"Wien peak for Sun ({T_sun} K): {lambda_max*1000:.1f} nm ({lambda_max:.3f} um)")
print("=> 태양의 peak는 가시광선 영역(~500 nm, 초록색)에 위치")

In [ ]:
# =============================================================================
# Part 5b: Beer-Lambert absorption and optical depth
# =============================================================================

wl_abs = np.linspace(380, 700, 2000)

# Define optical depth profiles for individual elements.
# Each element has absorption at specific wavelengths.
elements = {
    "Na (Sodium)": {
        "lines": [588.99, 589.59],
        "tau_max": [3.5, 3.0],
        "width": [0.5, 0.5],
        "color": "orange",
    },
    "H (Hydrogen)": {
        "lines": [656.28, 486.13],
        "tau_max": [2.5, 2.0],
        "width": [0.8, 0.6],
        "color": "red",
    },
    "Ca II (Calcium)": {
        "lines": [393.37, 396.85],
        "tau_max": [3.0, 2.8],
        "width": [0.7, 0.7],
        "color": "purple",
    },
    "Fe (Iron)": {
        "lines": [430.78, 527.04],
        "tau_max": [1.8, 1.5],
        "width": [0.6, 0.5],
        "color": "gray",
    },
}

fig, axes = plt.subplots(3, 1, figsize=(14, 12), gridspec_kw={"height_ratios": [2, 2, 3]})

# Top: Optical depth tau(lambda) for each element.
ax_tau = axes[0]
tau_total = np.zeros_like(wl_abs)

for elem_name, elem_data in elements.items():
    tau_elem = np.zeros_like(wl_abs)
    for wl0, tau_max, width in zip(elem_data["lines"], elem_data["tau_max"],
                                    elem_data["width"]):
        tau_elem += tau_max * np.exp(-0.5 * ((wl_abs - wl0) / width)**2)
    tau_total += tau_elem
    ax_tau.plot(wl_abs, tau_elem, color=elem_data["color"], linewidth=1.5,
                label=elem_name, alpha=0.8)

ax_tau.plot(wl_abs, tau_total, "k-", linewidth=2, label="Total $\\tau$", alpha=0.5)
ax_tau.set_ylabel("Optical Depth $\\tau(\\lambda)$\n광학 깊이", fontsize=12)
ax_tau.set_title("Optical Depth by Element\n원소별 광학 깊이", fontsize=13)
ax_tau.legend(fontsize=9, ncol=3)
ax_tau.set_xlim(380, 700)

# Middle: Transmission = exp(-tau).
ax_trans = axes[1]
transmission = np.exp(-tau_total)
ax_trans.plot(wl_abs, transmission, "k-", linewidth=1.5)
ax_trans.fill_between(wl_abs, transmission, 1, alpha=0.2, color="red",
                       label="Absorbed / 흡수된 부분")
ax_trans.fill_between(wl_abs, 0, transmission, alpha=0.2, color="green",
                       label="Transmitted / 투과된 부분")
ax_trans.set_ylabel("Transmission $e^{-\\tau}$\n투과율", fontsize=12)
ax_trans.set_title("Beer-Lambert Law: $I = I_0 \\cdot e^{-\\tau(\\lambda)}$\n"
                   "Beer-Lambert 법칙", fontsize=13)
ax_trans.legend(fontsize=10)
ax_trans.set_xlim(380, 700)
ax_trans.set_ylim(0, 1.05)

# Bottom: Resulting solar spectrum.
ax_spec = axes[2]

# Blackbody continuum.
wl_m_abs = wl_abs * 1e-9
bb_abs = planck_wavelength(wl_m_abs, T_sun)
bb_abs_norm = bb_abs / bb_abs.max()

# Absorbed spectrum.
solar_abs = bb_abs_norm * transmission

ax_spec.fill_between(wl_abs, solar_abs, alpha=0.3, color="gray")
ax_spec.plot(wl_abs, bb_abs_norm, "k--", alpha=0.4, linewidth=1, label="Blackbody continuum")
ax_spec.plot(wl_abs, solar_abs, "k-", linewidth=0.8, label="After absorption")

# Label lines with element names.
for elem_name, elem_data in elements.items():
    for wl0 in elem_data["lines"]:
        if 380 <= wl0 <= 700:
            ax_spec.axvline(wl0, color=elem_data["color"], alpha=0.5,
                           linewidth=0.8, linestyle="--")

# Color strip at bottom.
for wl_i in wl_abs[::3]:
    rgb = wavelength_to_rgb(wl_i)
    idx = np.argmin(np.abs(wl_abs - wl_i))
    darkened = tuple(ch * solar_abs[idx] for ch in rgb)
    ax_spec.axvspan(wl_i - 0.3, wl_i + 0.3, ymin=0, ymax=0.05, color=darkened)

ax_spec.set_xlabel("Wavelength / 파장 (nm)", fontsize=12)
ax_spec.set_ylabel("Normalized Intensity\n정규화 강도", fontsize=12)
ax_spec.set_title("Resulting Solar Spectrum After Absorption\n"
                   "흡수 후 태양 스펙트럼", fontsize=13)
ax_spec.legend(fontsize=10)
ax_spec.set_xlim(380, 700)
ax_spec.set_ylim(0, 1.15)

plt.tight_layout()
plt.show()

print("\n=> 각 원소가 고유한 파장에서 흡수를 일으켜 어두운 선이 생김")
print("=> 이 원리는 Kirchhoff & Bunsen (1859)이 밝혔지만,")
print("   Fraunhofer가 관측한 현상이 그 기초를 제공함")

---
## Part 6: Stellar Spectra Comparison
## 파트 6: 항성 스펙트럼 비교

Fraunhofer는 태양뿐 아니라 **밝은 항성들(Sirius, Castor, Betelgeuse 등)**의 스펙트럼도 관측했습니다.
그는 놀라운 사실을 발견했습니다:

> "별마다 흡수선의 패턴이 다르다!"

이것은 매우 중요한 발견이었습니다:
- 별들이 **서로 다른 화학 조성**을 가지고 있다는 최초의 관측적 증거
- 별들이 **서로 다른 온도**를 가지고 있다는 암시
- 나중에 Harvard spectral classification (O, B, A, F, G, K, M)의 기초가 됨

**현대적 이해**:
- 온도가 다르면 같은 원소라도 다른 이온화 상태 -> 다른 흡수선 패턴
- 조성이 다르면 특정 원소의 선이 강하거나 약함
- Spectral type은 주로 **온도**에 의해 결정됨 (O: hot -> M: cool)

In [ ]:
# =============================================================================
# Part 6: Stellar spectra comparison
# =============================================================================

def simulate_stellar_spectrum(wl_nm: np.ndarray, T: float,
                              line_strengths: dict,
                              width_nm: float = 0.8) -> np.ndarray:
    """Simulate a stellar spectrum with absorption lines.

    Args:
        wl_nm: Wavelength array in nm.
        T: Stellar effective temperature in Kelvin.
        line_strengths: Dict mapping line wavelength (nm) to absorption strength.
        width_nm: Width of absorption lines in nm.

    Returns:
        Normalized spectrum array.
    """
    wl_m = wl_nm * 1e-9
    bb = planck_wavelength(wl_m, T)
    bb_norm = bb / bb.max()

    # Apply absorption lines.
    for wl0, strength in line_strengths.items():
        profile = strength * np.exp(-0.5 * ((wl_nm - wl0) / width_nm)**2)
        bb_norm *= (1 - profile)

    return bb_norm


# Define stellar types with different temperatures and line patterns.
# Line strengths vary with temperature due to ionization physics.
stars = {
    "Sirius (A1V, ~9940 K)\nFraunhofer observed strong H lines": {
        "T": 9940,
        "lines": {
            # Strong Hydrogen Balmer lines (dominant in A-type stars).
            656.28: 0.90,  # H-alpha
            486.13: 0.85,  # H-beta
            434.05: 0.80,  # H-gamma
            410.17: 0.75,  # H-delta
            # Weak metals.
            393.37: 0.25,  # Ca K
            396.85: 0.20,  # Ca H
            589.30: 0.10,  # Na D
        },
        "color": "blue",
    },
    "Sun (G2V, 5778 K)\nFraunhofer's primary target": {
        "T": 5778,
        "lines": {
            # Moderate H lines.
            656.28: 0.70,
            486.13: 0.60,
            # Strong metal lines.
            393.37: 0.85,  # Ca K
            396.85: 0.80,  # Ca H
            589.30: 0.90,  # Na D (doublet)
            589.59: 0.85,
            527.04: 0.55,  # Fe
            430.78: 0.60,  # Fe/Ca
        },
        "color": "goldenrod",
    },
    "Betelgeuse (M2Iab, ~3600 K)\nFraunhofer noted very different pattern": {
        "T": 3600,
        "lines": {
            # Weak H lines (too cool for strong H excitation).
            656.28: 0.30,
            486.13: 0.20,
            # Very strong metal and molecular lines.
            393.37: 0.70,  # Ca K
            396.85: 0.65,  # Ca H
            589.30: 0.85,  # Na D
            589.59: 0.80,
            527.04: 0.70,  # Fe
            430.78: 0.65,  # Fe/Ca
            # TiO molecular bands (characteristic of M stars).
            505.0: 0.50,
            550.0: 0.40,
            620.0: 0.45,
            670.0: 0.55,
        },
        "color": "red",
    },
    "Vega (A0V, ~9600 K)\nSpectral standard star": {
        "T": 9600,
        "lines": {
            # Very strong H Balmer lines (classic A0 star).
            656.28: 0.92,
            486.13: 0.88,
            434.05: 0.85,
            410.17: 0.80,
            397.00: 0.75,
            # Very weak metals.
            393.37: 0.15,
            589.30: 0.05,
        },
        "color": "steelblue",
    },
}

wl_star = np.linspace(380, 720, 2000)

fig, axes = plt.subplots(len(stars), 1, figsize=(14, 3.5 * len(stars)),
                          sharex=True)

for idx, (star_name, star_data) in enumerate(stars.items()):
    ax = axes[idx]
    spectrum = simulate_stellar_spectrum(wl_star, star_data["T"],
                                         star_data["lines"])
    ax.fill_between(wl_star, spectrum, alpha=0.2, color=star_data["color"])
    ax.plot(wl_star, spectrum, color=star_data["color"], linewidth=0.8)

    # Add color strip.
    for wl_i in wl_star[::5]:
        rgb = wavelength_to_rgb(wl_i)
        i_idx = np.argmin(np.abs(wl_star - wl_i))
        darkened = tuple(ch * spectrum[i_idx] for ch in rgb)
        ax.axvspan(wl_i - 0.5, wl_i + 0.5, ymin=0, ymax=0.06, color=darkened)

    ax.set_ylabel("Intensity\n강도", fontsize=10)
    ax.set_title(star_name, fontsize=12, color=star_data["color"],
                 fontweight="bold")
    ax.set_ylim(0, 1.1)
    ax.set_xlim(380, 720)

    # Mark key lines.
    key_lines = {"H-a": 656.28, "H-b": 486.13, "Ca K": 393.37,
                 "Na D": 589.30}
    for lname, lwl in key_lines.items():
        if lwl in star_data["lines"] and star_data["lines"][lwl] > 0.3:
            ax.axvline(lwl, color="gray", alpha=0.3, linewidth=0.5)
            ax.text(lwl, 1.02, lname, fontsize=7, ha="center", color="gray")

axes[-1].set_xlabel("Wavelength / 파장 (nm)", fontsize=12)

fig.suptitle("Stellar Spectra Comparison: Different Stars, Different Line Patterns\n"
             "항성 스펙트럼 비교: 별마다 다른 흡수선 패턴 "
             "(Fraunhofer's Key Observation)",
             fontsize=14, fontweight="bold", y=1.01)
plt.tight_layout()
plt.show()

print("\n=== Key Observations / 핵심 관측 결과 ===")
print("1. A-type stars (Sirius, Vega): 매우 강한 H Balmer 선, 약한 금속선")
print("   -> 높은 온도에서 수소 원자가 n=2 준위에 많이 여기됨")
print("2. G-type star (Sun): H선과 금속선 모두 중간 강도")
print("   -> 중간 온도에서 다양한 원소가 흡수에 기여")
print("3. M-type star (Betelgeuse): 약한 H선, 강한 금속선 + TiO 분자대")
print("   -> 낮은 온도에서 분자가 형성되고, H 여기 부족")
print("\n=> Fraunhofer의 관측은 항성 분류학의 시작이었음!")

---
## Summary / 요약

### Fraunhofer's Key Contributions / Fraunhofer의 핵심 기여

| Contribution / 기여 | Significance / 의의 | Modern Impact / 현대적 영향 |
|---|---|---|
| 574+ dark lines cataloged / 574개 이상의 어두운 선 분류 | First systematic solar spectral atlas / 최초의 체계적 태양 스펙트럼 도감 | Foundation of spectroscopy / 분광학의 기초 |
| A-K labeling system / A-K 명명 체계 | Standard reference wavelengths / 표준 기준 파장 확립 | Still used today (Na D, Ca H&K) / 오늘날에도 사용 |
| Minimum deviation method / 최소 편향각 측정법 | Precision measurement of n(lambda) / 굴절률의 정밀 측정 | Optical glass characterization / 광학 유리 특성 평가 |
| Diffraction grating / 회절격자 발명 | Absolute wavelength measurement / 절대 파장 측정 가능 | Modern spectrographs / 현대 분광기의 원형 |
| Stellar spectra differences / 항성 스펙트럼 차이 발견 | Stars have different compositions / 별들의 조성 차이 | Spectral classification (OBAFGKM) / 항성 분류 |

### Connection to Next Papers / 후속 논문과의 연결

| Next Paper / 다음 논문 | Connection / 연결점 |
|---|---|
| **Kirchhoff & Bunsen (1859-1860)** | Fraunhofer 선의 물리적 원인 규명: 원소의 방출/흡수 스펙트럼 일치 법칙 |
| **Kirchhoff's Laws of Radiation (1860)** | 흑체 복사와 흡수/방출의 관계 정립 (Kirchhoff's 3 laws) |
| **Angstrom (1868)** | Fraunhofer 선의 정밀 파장 측정, Angstrom 단위 도입 |
| **Rowland (1895)** | 회절격자 기술 발전, 14,000개 이상의 태양 흡수선 카탈로그 |
| **Hale (1892)** | Spectroheliograph 발명 -- 특정 Fraunhofer 선으로 태양 촬영 |

### Key Equations Summary / 핵심 공식 정리

| Equation / 공식 | Description / 설명 |
|---|---|
| $n(\lambda) = A + B/\lambda^2 + C/\lambda^4$ | Cauchy equation for dispersion / 코시 분산 방정식 |
| $n = \sin\left(\frac{\alpha + \delta_{\min}}{2}\right) / \sin\left(\frac{\alpha}{2}\right)$ | Minimum deviation formula / 최소 편향각 공식 |
| $d \sin\theta = m\lambda$ | Grating equation / 격자 방정식 |
| $R = mN$ | Grating resolving power / 격자 분해능 |
| $B(\lambda,T) = \frac{2hc^2}{\lambda^5} \cdot \frac{1}{e^{hc/\lambda k_B T}-1}$ | Planck function / Planck 함수 |
| $I(\lambda) = I_0(\lambda) \cdot e^{-\tau(\lambda)}$ | Beer-Lambert law / 흡수 법칙 |